# RNAAPIA — Fine-Tuning Dataset Organizacional

Parte do modelo pré-treinado (v9) e adapta-o às 5 pessoas da organização.

```
0. Setup
1. MTCNN — alinhar fotos da organização
2. Data Augmentation (~20x por foto)
3. DataLoaders
4. Fine-tuning FaceCNN
5. Fine-tuning ResNet-50
6. Avaliação FAR / FRR / EER
7. Guardar modelos e gallery
```

## ⚠️ Antes de começar
1. **Add Data** → Upload → pasta `Dataset_Organização/` com subpastas por pessoa
2. **Add Data** → Models → `facecnn` (v2) e `resnet-50` (v2)
3. **Settings** → Accelerator → GPU T4

**Estrutura esperada:**
```
Dataset_Organização/
    pessoa1/   ← 10 fotos .jpg/.png
    pessoa2/
    pessoa3/
    pessoa4/
    pessoa5/
```

---
## 0. Setup

In [ ]:
import torch, os, glob, json, time, shutil
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from copy import deepcopy

print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
%%capture
!pip install facenet-pytorch==2.5.3 scikit-learn matplotlib tqdm
print('✓ Dependências instaladas')

In [ ]:
BASE = '/kaggle/working'

PATHS = {
    'org_raw'     : '/kaggle/input/Dataset_Organização',
    'org_aligned' : f'{BASE}/org_aligned',
    'org_aug'     : f'{BASE}/org_augmented',
    'checkpoints' : f'{BASE}/checkpoints',
    'logs'        : f'{BASE}/logs',
}
for k, v in PATHS.items():
    if k != 'org_raw':
        os.makedirs(v, exist_ok=True)

# Checkpoints v9
CNN_CKPT    = '/kaggle/input/models/goncalojesus/facecnn/pytorch/default/2/best_cnn_v9.pth'
RESNET_CKPT = '/kaggle/input/models/goncalojesus/resnet-50/pytorch/default/2/best_resnet_v9.pth'

# Verificar
print('A verificar checkpoints:')
for name, path in [('FaceCNN v9', CNN_CKPT), ('ResNet-50 v9', RESNET_CKPT)]:
    print(f'  {"✓" if os.path.exists(path) else "⚠️  NÃO ENCONTRADO"} {name}: {path}')

# Verificar dataset
print('\nDataset organizacional:')
if os.path.exists(PATHS['org_raw']):
    people = sorted([d for d in os.listdir(PATHS['org_raw'])
                     if os.path.isdir(os.path.join(PATHS['org_raw'], d))])
    for p in people:
        imgs = (glob.glob(os.path.join(PATHS['org_raw'], p, '*.jpg')) +
                glob.glob(os.path.join(PATHS['org_raw'], p, '*.jpeg')) +
                glob.glob(os.path.join(PATHS['org_raw'], p, '*.png')))
        print(f'  {p}: {len(imgs)} imagens')
else:
    print(f'  ⚠️  Pasta não encontrada: {PATHS["org_raw"]}')
    print('  Verifica o nome exato no painel Input do Kaggle')

# Hiperparâmetros de fine-tuning (melhores do v9)
IMAGE_SIZE     = 192
EMBEDDING_SIZE = 512
LR             = 1e-3   # LR mais alto que v9 — dataset pequeno, adaptar rápido
WEIGHT_DECAY   = 1e-4
LABEL_SMOOTH   = 0.05
BATCH_SIZE     = 16     # batch pequeno por causa do dataset pequeno
FT_EPOCHS      = 50
N_AUGMENTS     = 20     # variações por foto original
DROPOUT        = 0.5

---
## 1. MTCNN — Deteção e Alinhamento
Deteta e alinha as faces nas fotos originais. Guarda em `org_aligned/`.

In [ ]:
from facenet_pytorch import MTCNN
from PIL import Image
from tqdm import tqdm

margin = max(20, IMAGE_SIZE // 6)
mtcnn  = MTCNN(
    image_size=IMAGE_SIZE,
    margin=margin,
    min_face_size=20,
    thresholds=[0.6, 0.7, 0.7],
    factor=0.709,
    post_process=False,
    device=DEVICE
)
print(f'✓ MTCNN — image_size={IMAGE_SIZE}px | margin={margin}px')

people = sorted([d for d in os.listdir(PATHS['org_raw'])
                 if os.path.isdir(os.path.join(PATHS['org_raw'], d))])
print(f'Pessoas: {people}\n')

total_saved, total_failed = 0, 0

for person in tqdm(people, desc='MTCNN'):
    out_dir = os.path.join(PATHS['org_aligned'], person)
    os.makedirs(out_dir, exist_ok=True)

    img_paths = (glob.glob(os.path.join(PATHS['org_raw'], person, '*.jpg'))  +
                 glob.glob(os.path.join(PATHS['org_raw'], person, '*.jpeg')) +
                 glob.glob(os.path.join(PATHS['org_raw'], person, '*.png'))  +
                 glob.glob(os.path.join(PATHS['org_raw'], person, '*.JPG'))  +
                 glob.glob(os.path.join(PATHS['org_raw'], person, '*.JPEG')))

    saved = 0
    for img_path in img_paths:
        try:
            face = mtcnn(Image.open(img_path).convert('RGB'))
            if face is not None:
                out_name = os.path.splitext(os.path.basename(img_path))[0] + '.jpg'
                Image.fromarray(face.permute(1,2,0).byte().numpy()).save(
                    os.path.join(out_dir, out_name))
                saved += 1
            else:
                print(f'  ⚠️  Face não detetada: {os.path.basename(img_path)}')
                total_failed += 1
        except Exception as e:
            print(f'  ✗ Erro em {os.path.basename(img_path)}: {e}')
            total_failed += 1

    total_saved += saved
    print(f'  {person}: {len(img_paths)} originais → {saved} faces alinhadas')

print(f'\n✓ MTCNN concluído: {total_saved} faces | {total_failed} falhas')

---
## 2. Data Augmentation
Gera ~20 variações por foto → ~200 imagens por pessoa.

In [ ]:
from torchvision import transforms
import random
random.seed(42)

aug_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.4, hue=0.1),
    transforms.RandomGrayscale(p=0.1),
    transforms.RandomApply([transforms.GaussianBlur(3)], p=0.3),
    transforms.RandomPerspective(distortion_scale=0.2, p=0.3),
])

for person in people:
    out_dir = os.path.join(PATHS['org_aug'], person)
    os.makedirs(out_dir, exist_ok=True)

    aligned = glob.glob(os.path.join(PATHS['org_aligned'], person, '*.jpg'))
    if not aligned:
        print(f'  ⚠️  {person}: sem faces alinhadas — MTCNN falhou para todas as fotos')
        continue

    for i, img_path in enumerate(aligned):
        img = Image.open(img_path).convert('RGB').resize((IMAGE_SIZE, IMAGE_SIZE))
        # Guardar original
        img.save(os.path.join(out_dir, f'orig_{i:03d}.jpg'))
        # Gerar augmentações
        for j in range(N_AUGMENTS):
            aug_transform(img).save(os.path.join(out_dir, f'aug_{i:03d}_{j:02d}.jpg'))

    total = len(os.listdir(out_dir))
    print(f'  {person}: {len(aligned)} originais → {total} imagens')

print('\n✓ Augmentation concluída')

---
## 3. DataLoaders

In [ ]:
from torch.utils.data import DataLoader, random_split
from torchvision import datasets

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.RandomApply([transforms.GaussianBlur(3)], p=0.1),
    transforms.RandomApply([transforms.RandomRotation(15)], p=0.3),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3),
    transforms.RandomErasing(p=0.2, scale=(0.02, 0.1)),
])
val_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

full_dataset = datasets.ImageFolder(PATHS['org_aug'], transform=train_transform)
NUM_CLASSES  = len(full_dataset.classes)
total        = len(full_dataset)
n_train      = int(0.70 * total)
n_val        = int(0.15 * total)
n_test       = total - n_train - n_val

train_set, val_set, test_set = random_split(
    full_dataset, [n_train, n_val, n_test],
    generator=torch.Generator().manual_seed(42)
)

# Val e test com transform sem augmentation
val_dataset  = datasets.ImageFolder(PATHS['org_aug'], transform=val_transform)
from torch.utils.data import Subset
val_loader   = DataLoader(Subset(val_dataset,  val_set.indices),
                          batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(Subset(val_dataset,  test_set.indices),
                          batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)

print(f'Pessoas: {full_dataset.classes}')
print(f'Total: {total} | Train: {n_train} | Val: {n_val} | Test: {n_test}')
print(f'Classes: {NUM_CLASSES}')

---
## 4. Definição dos Modelos

In [ ]:
import torchvision.models as tv_models

class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True), nn.MaxPool2d(2))
    def forward(self, x): return self.block(x)

class FaceCNN(nn.Module):
    def __init__(self, num_classes, embedding_size=512, image_size=192, dropout=0.5):
        super().__init__()
        self.conv_blocks = nn.Sequential(
            ConvBlock(3, 32), ConvBlock(32, 64),
            ConvBlock(64, 128), ConvBlock(128, 256))
        fm = image_size // 16
        self.embedding = nn.Sequential(
            nn.Flatten(), nn.Dropout(p=dropout),
            nn.Linear(256 * fm * fm, embedding_size),
            nn.BatchNorm1d(embedding_size), nn.ReLU(inplace=True))
        self.classifier = nn.Linear(embedding_size, num_classes)
    def forward(self, x, return_embedding=False):
        emb = self.embedding(self.conv_blocks(x))
        return F.normalize(emb, dim=1) if return_embedding else self.classifier(emb)

class ResNet50Face(nn.Module):
    def __init__(self, num_classes, embedding_size=512, dropout=0.5):
        super().__init__()
        backbone      = tv_models.resnet50(weights=None)
        self.features = nn.Sequential(*list(backbone.children())[:-1])
        self.embedding = nn.Sequential(
            nn.Flatten(), nn.Dropout(p=dropout),
            nn.Linear(2048, embedding_size),
            nn.BatchNorm1d(embedding_size), nn.ReLU(inplace=True))
        self.classifier = nn.Linear(embedding_size, num_classes)
    def forward(self, x, return_embedding=False):
        emb = self.embedding(self.features(x))
        return F.normalize(emb, dim=1) if return_embedding else self.classifier(emb)


def load_for_finetuning(ModelClass, ckpt_path, num_classes, **kwargs):
    """
    Carrega o modelo pré-treinado, substitui o classifier pelo número
    de classes organizacionais, e congela as camadas convolucionais.
    Apenas embedding + classifier são treináveis.
    """
    # Carregar arquitetura com o número de classes do pré-treino
    # (não sabemos exatamente, mas o load parcial trata disso)
    ckpt       = torch.load(ckpt_path, map_location=DEVICE)
    old_state  = ckpt['model']

    # Descobrir n_classes do checkpoint pelo classifier.weight
    old_n_classes = old_state['classifier.weight'].shape[0]

    # Instanciar com as classes antigas para carregar os pesos
    model = ModelClass(num_classes=old_n_classes, **kwargs).to(DEVICE)
    model.load_state_dict(old_state)

    # Substituir classifier pelo número de classes organizacionais
    model.classifier = nn.Linear(EMBEDDING_SIZE, num_classes).to(DEVICE)

    # Congelar conv_blocks / features — treinar só embedding + classifier
    for name, param in model.named_parameters():
        param.requires_grad = any(l in name for l in ['embedding', 'classifier'])

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_p   = sum(p.numel() for p in model.parameters())
    print(f'  Pesos carregados de {os.path.basename(ckpt_path)}')
    print(f'  Classes: {old_n_classes} → {num_classes}')
    print(f'  Parâmetros treináveis: {trainable:,} / {total_p:,}')
    return model


print('A carregar modelos...')
print('\nFaceCNN:')
cnn_model = load_for_finetuning(
    FaceCNN, CNN_CKPT, NUM_CLASSES,
    embedding_size=EMBEDDING_SIZE, image_size=IMAGE_SIZE, dropout=DROPOUT)

print('\nResNet-50:')
rn_model = load_for_finetuning(
    ResNet50Face, RESNET_CKPT, NUM_CLASSES,
    embedding_size=EMBEDDING_SIZE, dropout=DROPOUT)

---
## 5. Funções de Treino

In [ ]:
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR

def train_one_epoch(model, loader, optimizer, scaler, label_smoothing=0.05):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            out  = model(imgs)
            loss = F.cross_entropy(out, labels, label_smoothing=label_smoothing)
        scaler.scale(loss).backward()
        scaler.step(optimizer); scaler.update()
        total_loss += loss.item() * imgs.size(0)
        correct    += (out.argmax(1) == labels).sum().item()
        total      += imgs.size(0)
    return total_loss / total, correct / total

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        out  = model(imgs).float()
        loss = F.cross_entropy(out, labels)
        if not torch.isnan(loss):
            total_loss += loss.item() * imgs.size(0)
        correct += (out.argmax(1) == labels).sum().item()
        total   += imgs.size(0)
    return total_loss / total, correct / total


def finetune(model, model_name, save_path):
    optimizer = Adam(
        [p for p in model.parameters() if p.requires_grad],
        lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = CosineAnnealingLR(optimizer, T_max=FT_EPOCHS, eta_min=1e-6)
    scaler    = torch.amp.GradScaler('cuda')
    history   = []
    best_val  = 0.0
    best_state = None

    print(f'\n── Fine-tuning {model_name} ({FT_EPOCHS} épocas) ──────────────────')
    for epoch in range(FT_EPOCHS):
        t0 = time.time()
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, scaler, LABEL_SMOOTH)
        val_loss,   val_acc   = evaluate(model, val_loader)
        scheduler.step()
        elapsed = time.time() - t0

        history.append({'epoch': epoch, 'train_acc': round(train_acc,4),
                        'val_acc': round(val_acc,4),
                        'train_loss': round(train_loss,4),
                        'val_loss': round(val_loss,4)})

        improved = ' ✓' if val_acc > best_val else ''
        print(f'[{model_name}] {epoch+1:02d}/{FT_EPOCHS} | '
              f'Train: {train_loss:.4f} ({train_acc*100:.1f}%) | '
              f'Val: {val_loss:.4f} ({val_acc*100:.1f}%){improved} | {elapsed:.0f}s')

        if val_acc > best_val:
            best_val   = val_acc
            best_state = deepcopy(model.state_dict())

    # Carregar melhor e avaliar no test set
    model.load_state_dict(best_state)
    _, test_acc = evaluate(model, test_loader)

    # Guardar
    torch.save({
        'model'    : best_state,
        'val_acc'  : best_val,
        'test_acc' : test_acc,
        'classes'  : full_dataset.classes,
        'history'  : history,
    }, save_path)

    print(f'\n✓ {model_name} concluído!')
    print(f'  Melhor val acc: {best_val*100:.2f}%')
    print(f'  Test acc:       {test_acc*100:.2f}%')
    return model, history, best_val, test_acc

print('✓ Funções prontas')

---
## 6. Fine-Tuning FaceCNN

In [ ]:
cnn_model, cnn_hist, cnn_val, cnn_test = finetune(
    cnn_model,
    'FaceCNN',
    f'{BASE}/best_cnn_org.pth'
)

---
## 7. Fine-Tuning ResNet-50

In [ ]:
rn_model, rn_hist, rn_val, rn_test = finetune(
    rn_model,
    'ResNet-50',
    f'{BASE}/best_resnet_org.pth'
)

---
## 8. Comparação + Curvas de Treino

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, key, ylabel in [
    (axes[0], 'val_loss', 'Loss'),
    (axes[1], 'val_acc',  'Accuracy (%)'),
]:
    scale = 100 if key == 'val_acc' else 1
    ax.plot([h['epoch']+1 for h in cnn_hist], [h[key]*scale for h in cnn_hist],
            'b-o', markersize=3, label='FaceCNN')
    ax.plot([h['epoch']+1 for h in rn_hist],  [h[key]*scale for h in rn_hist],
            'r-o', markersize=3, label='ResNet-50')
    ax.set_xlabel('Epoch'); ax.set_ylabel(ylabel)
    ax.set_title(f'{ylabel} — Fine-tuning Organizacional')
    ax.legend(); ax.grid(True, alpha=0.3)

plt.suptitle(f'Fine-tuning — {NUM_CLASSES} pessoas | {n_train} imgs treino', fontsize=12)
plt.tight_layout()
plt.savefig(f'{BASE}/finetuning_curves.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n── Resumo ─────────────────────────────────────────────')
print(f'{"":20s} {"FaceCNN":>10s} {"ResNet-50":>10s}')
print(f'{"Melhor val acc":20s} {cnn_val*100:>9.2f}% {rn_val*100:>9.2f}%')
print(f'{"Test acc":20s} {cnn_test*100:>9.2f}% {rn_test*100:>9.2f}%')
cnn_gap = (max(h['train_acc'] for h in cnn_hist) - cnn_val) * 100
rn_gap  = (max(h['train_acc'] for h in rn_hist)  - rn_val)  * 100
print(f'{"Gap train/val":20s} {cnn_gap:>+9.2f}% {rn_gap:>+9.2f}%')

---
## 9. Avaliação FAR / FRR / EER
Avalia ambos os modelos no modo verificação (embedding + cosine similarity).

In [ ]:
from sklearn.metrics import classification_report

def compute_far_frr(model, model_name):
    model.eval()

    # Embeddings do test set
    all_emb, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in test_loader:
            emb = model(imgs.to(DEVICE), return_embedding=True)
            all_emb.append(emb.cpu()); all_labels.append(labels)
    all_emb    = torch.cat(all_emb)
    all_labels = torch.cat(all_labels)

    # Gallery: embedding médio por pessoa (calculado no train set)
    gallery = {}
    with torch.no_grad():
        for imgs, labels in train_loader:
            emb = model(imgs.to(DEVICE), return_embedding=True).cpu()
            for e, l in zip(emb, labels):
                gallery.setdefault(l.item(), []).append(e)

    gallery_means  = {k: F.normalize(torch.stack(v).mean(0).unsqueeze(0), dim=1).squeeze()
                      for k, v in gallery.items()}
    gallery_matrix = torch.stack([gallery_means[i] for i in range(NUM_CLASSES)])

    similarities   = all_emb @ gallery_matrix.T
    max_scores     = similarities.max(dim=1).values.numpy()
    pred_labels    = similarities.argmax(dim=1).numpy()
    true_labels    = all_labels.numpy()

    genuine_scores  = max_scores[pred_labels == true_labels]
    impostor_scores = max_scores[pred_labels != true_labels]

    thresholds = np.linspace(0, 1, 1000)
    FARs = np.array([(impostor_scores >= t).mean() if len(impostor_scores) > 0 else 0.0 for t in thresholds])
    FRRs = np.array([(genuine_scores  <  t).mean() if len(genuine_scores)  > 0 else 0.0 for t in thresholds])
    eer_idx = np.argmin(np.abs(FARs - FRRs))
    EER     = (FARs[eer_idx] + FRRs[eer_idx]) / 2
    EER_t   = thresholds[eer_idx]

    print(f'\n── {model_name} ─────────────────────────────────────')
    print(f'  EER:           {EER*100:.2f}%')
    print(f'  Threshold EER: {EER_t:.3f}')
    print(f'  FAR @ EER:     {FARs[eer_idx]*100:.2f}%')
    print(f'  FRR @ EER:     {FRRs[eer_idx]*100:.2f}%')

    return EER, EER_t, FARs, FRRs, thresholds, genuine_scores, impostor_scores, gallery_means


cnn_eer, cnn_t, cnn_FARs, cnn_FRRs, thresholds, cnn_gen, cnn_imp, cnn_gallery = compute_far_frr(cnn_model, 'FaceCNN')
rn_eer,  rn_t,  rn_FARs,  rn_FRRs,  _,          rn_gen,  rn_imp,  rn_gallery  = compute_far_frr(rn_model,  'ResNet-50')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# FAR/FRR curves
for ax, FARs, FRRs, eer, eer_t, name, color in [
    (axes[0,0], cnn_FARs, cnn_FRRs, cnn_eer, cnn_t, 'FaceCNN',   'blue'),
    (axes[0,1], rn_FARs,  rn_FRRs,  rn_eer,  rn_t,  'ResNet-50', 'red'),
]:
    ax.plot(thresholds, FARs*100, 'r-',  linewidth=2, label='FAR (%)')
    ax.plot(thresholds, FRRs*100, 'b-',  linewidth=2, label='FRR (%)')
    ax.axvline(eer_t, color='green', linestyle='--',
               label=f'EER={eer*100:.2f}% @ t={eer_t:.2f}')
    ax.set_xlabel('Threshold'); ax.set_ylabel('Rate (%)')
    ax.set_title(f'{name} — FAR / FRR'); ax.legend(); ax.grid(True, alpha=0.3)

# Score distributions
for ax, gen, imp, eer_t, name in [
    (axes[1,0], cnn_gen, cnn_imp, cnn_t, 'FaceCNN'),
    (axes[1,1], rn_gen,  rn_imp,  rn_t,  'ResNet-50'),
]:
    ax.hist(gen, bins=20, alpha=0.6, color='green', label='Genuine',  density=True)
    ax.hist(imp, bins=20, alpha=0.6, color='red',   label='Impostor', density=True)
    ax.axvline(eer_t, color='black', linestyle='--', label='EER threshold')
    ax.set_xlabel('Cosine Similarity'); ax.set_title(f'{name} — Score Distribution')
    ax.legend(); ax.grid(True, alpha=0.3)

plt.suptitle('Avaliação Biométrica — Dataset Organizacional', fontsize=13)
plt.tight_layout()
plt.savefig(f'{BASE}/far_frr_org.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n── Resumo Biométrico ───────────────────────────────────')
print(f'{"":20s} {"FaceCNN":>10s} {"ResNet-50":>10s}')
print(f'{"EER":20s} {cnn_eer*100:>9.2f}% {rn_eer*100:>9.2f}%')
print(f'{"Threshold EER":20s} {cnn_t:>10.3f} {rn_t:>10.3f}')

---
## 10. Guardar Gallery e Modelos Finais
Guarda as galleries (embeddings médios por pessoa) e publica os modelos no Kaggle.

In [ ]:
import subprocess

# Guardar galleries com nomes das pessoas
cnn_gallery_named = {full_dataset.classes[k]: v for k, v in cnn_gallery.items()}
rn_gallery_named  = {full_dataset.classes[k]: v for k, v in rn_gallery.items()}

torch.save({'gallery': cnn_gallery_named, 'threshold': cnn_t,
            'classes': full_dataset.classes, 'image_size': IMAGE_SIZE},
           f'{BASE}/gallery_cnn.pt')
torch.save({'gallery': rn_gallery_named, 'threshold': rn_t,
            'classes': full_dataset.classes, 'image_size': IMAGE_SIZE},
           f'{BASE}/gallery_resnet.pt')
print('✓ gallery_cnn.pt e gallery_resnet.pt guardados')

# Publicar modelos no Kaggle
for model_slug, filename in [
    ('facecnn',   'best_cnn_org.pth'),
    ('resnet-50', 'best_resnet_org.pth'),
]:
    model_dir = f'{BASE}/upload_{model_slug}_org'
    os.makedirs(model_dir, exist_ok=True)
    shutil.copy(f'{BASE}/{filename}', f'{model_dir}/{filename}')

    result = subprocess.run([
        'kaggle', 'models', 'instances', 'versions', 'create',
        f'goncalojesus/{model_slug}/pytorch/default',
        '-p', model_dir
    ], capture_output=True, text=True)
    print(f'  {filename}: {(result.stdout or result.stderr).strip()[:80]}')

print(f'\n📁 Ficheiros para download no painel Output:')
print('  best_cnn_org.pth       ← FaceCNN fine-tuned')
print('  best_resnet_org.pth    ← ResNet-50 fine-tuned')
print('  gallery_cnn.pt         ← embeddings + threshold FaceCNN  ← DEMO')
print('  gallery_resnet.pt      ← embeddings + threshold ResNet    ← DEMO')
print('  finetuning_curves.png')
print('  far_frr_org.png')

---
## 11. Script de Demo Local (Webcam)
Gera o script para correr localmente com a câmera.

In [ ]:
# Usar o melhor modelo entre os dois
best_model_name = 'ResNet-50' if rn_eer <= cnn_eer else 'FaceCNN'
best_gallery    = 'gallery_resnet.pt' if rn_eer <= cnn_eer else 'gallery_cnn.pt'
best_ckpt       = 'best_resnet_org.pth' if rn_eer <= cnn_eer else 'best_cnn_org.pth'
use_resnet      = rn_eer <= cnn_eer

print(f'Modelo selecionado para demo: {best_model_name} (EER={min(rn_eer,cnn_eer)*100:.2f}%)')

script = f'''#!/usr/bin/env python3
# demo_acesso.py — Sistema de Controlo de Acesso Facial
# pip install torch torchvision facenet-pytorch opencv-python
# Ficheiros necessários: {best_ckpt} + {best_gallery}

import torch, cv2
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
from facenet_pytorch import MTCNN
from PIL import Image

CHECKPOINT  = "{best_ckpt}"
GALLERY_PT  = "{best_gallery}"
IMAGE_SIZE  = {IMAGE_SIZE}
USE_RESNET  = {use_resnet}

class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True), nn.MaxPool2d(2))
    def forward(self, x): return self.block(x)

class FaceCNN(nn.Module):
    def __init__(self, num_classes, emb=512, sz=192):
        super().__init__()
        self.conv_blocks = nn.Sequential(
            ConvBlock(3,32), ConvBlock(32,64), ConvBlock(64,128), ConvBlock(128,256))
        fm = sz // 16
        self.embedding = nn.Sequential(
            nn.Flatten(), nn.Dropout(0.5),
            nn.Linear(256*fm*fm, emb), nn.BatchNorm1d(emb), nn.ReLU(inplace=True))
        self.classifier = nn.Linear(emb, num_classes)
    def forward(self, x, return_embedding=False):
        e = self.embedding(self.conv_blocks(x))
        return F.normalize(e, dim=1) if return_embedding else self.classifier(e)

import torchvision.models as tv_models
class ResNet50Face(nn.Module):
    def __init__(self, num_classes, emb=512):
        super().__init__()
        backbone      = tv_models.resnet50(weights=None)
        self.features = nn.Sequential(*list(backbone.children())[:-1])
        self.embedding = nn.Sequential(
            nn.Flatten(), nn.Dropout(0.5),
            nn.Linear(2048, emb), nn.BatchNorm1d(emb), nn.ReLU(inplace=True))
        self.classifier = nn.Linear(emb, num_classes)
    def forward(self, x, return_embedding=False):
        e = self.embedding(self.features(x))
        return F.normalize(e, dim=1) if return_embedding else self.classifier(e)

device  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
gallery_data = torch.load(GALLERY_PT, map_location=device)
gallery = gallery_data["gallery"]
THRESHOLD = gallery_data["threshold"]
people  = list(gallery.keys())
gmat    = torch.stack([gallery[p] for p in people]).to(device)

ckpt        = torch.load(CHECKPOINT, map_location=device)
n_classes   = len(people)
ModelClass  = ResNet50Face if USE_RESNET else FaceCNN
model       = ModelClass(num_classes=n_classes).to(device)
model.load_state_dict(ckpt["model"])
model.eval()

mtcnn = MTCNN(image_size=IMAGE_SIZE, margin=IMAGE_SIZE//6, device=device)
tf    = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)])

print(f"Sistema iniciado ({{device}}) | Modelo: {best_model_name} | EER threshold: {{THRESHOLD:.3f}}")
print(f"Pessoas registadas: {{people}}")
print("Prima Q para sair.")

cap = cv2.VideoCapture(0)
while True:
    ret, frame = cap.read()
    if not ret: break
    pil = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    boxes, probs = mtcnn.detect(pil)
    if boxes is not None:
        for box, prob in zip(boxes, probs):
            if prob is None or prob < 0.9: continue
            face = mtcnn(pil)
            if face is None: continue
            t = tf(Image.fromarray(face.permute(1,2,0).byte().numpy())).unsqueeze(0).to(device)
            with torch.no_grad():
                emb   = model(t, return_embedding=True)
                sims  = (emb @ gmat.T).squeeze()
                score = sims.max().item()
                idx   = sims.argmax().item()
            x1,y1,x2,y2 = [int(b) for b in box]
            if score >= THRESHOLD:
                color = (0, 255, 0)
                label = f"PERMITIDO: {{people[idx]}} ({{score:.2f}})"
            else:
                color = (0, 0, 255)
                label = f"NEGADO ({{score:.2f}})"
            cv2.rectangle(frame, (x1,y1), (x2,y2), color, 2)
            cv2.putText(frame, label, (x1, y1-10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.65, color, 2)
    cv2.imshow("Controlo de Acesso — RNAAPIA", frame)
    if cv2.waitKey(1) & 0xFF == ord("q"): break

cap.release()
cv2.destroyAllWindows()
'''

with open(f'{BASE}/demo_acesso.py', 'w') as f:
    f.write(script)
print('✓ demo_acesso.py gerado')
print(f'\nPara correr localmente:')
print('  1. Descarrega best_*_org.pth e gallery_*.pt do painel Output')
print('  2. pip install torch torchvision facenet-pytorch opencv-python')
print('  3. python demo_acesso.py')